### Research Question 1

**Can downprojection reveal systematic patterns in the model’s false-positive and false-negative predictions?**

In [3]:
import numpy as np
import pandas as pd
import plotly.express as px
import plotly.graph_objects as go
from plotly.subplots import make_subplots
from sklearn.neighbors import NearestNeighbors

In [4]:
# Import data from the base notebook
import import_ipynb
from IPython.utils.capture import capture_output

In [5]:
# Use "capture" do suppress plotting images
with capture_output() as captured:
    %run xraymodel.ipynb

In [ ]:
visualization_functions = [
    visualize_space_traces,
    visualize_space_prediction_scores_traces,
    visualize_classification_outcome_traces
]

visualization_names = [
    "Groundtruth with Gender",
    "Model Confidence Score",
    "Model Classification outcomes",
]
# NOTE: This creates heavy plot, execute this function only if really needed.
create_multi_pathology_viewer(
        downprojector=downprojector,
        projection_method="S-UMAP",
        visualization_fns=visualization_functions,
        visualization_names= visualization_names,
        model_score_predictions=MODEL_SCORE_PREDICTIONS_CONVERTED,
        ground_truth=XRAY_LABELS_LIST,
        dataset_label_indices_map=DATASET_LABEL_INDICES_MAP,
        dataset_gender_list=XRAY_GENDER_LIST,
        dataset_ages=XRAY_AGE_LIST,
        threshold=0.7
)

### Observation

After the downprojection and visualization of the pathology labels, two distinct types of clustering patterns can be observed.

The first type consists of pathologies where the positive and negative samples form two clearly distinguishable clusters. This pattern can be observed for Atelectasis, Cardiomegaly, Edema, Lung Opacity, and Effusion. For these pathologies, the model generally performed relatively well compared with the other findings, particularly for Edema, Effusion, Lung Opacity, and Cardiomegaly. The clear separation between positive and negative samples in the projection appears to correspond to the model's comparatively stronger ability to distinguish between the two classes.

The second type consists of a large manifold surrounded by smaller clusters or individual samples. In most cases, the large central manifold primarily contains the negative cases, while the smaller satellite clusters consist mainly of positive cases. This pattern can be observed for Pneumonia, Consolidation, Lung Lesion, Pneumothorax, and Fracture. For these pathologies, the model performed considerably worse. In some cases, it was barely able to detect any positive cases, suggesting that the positive samples are less clearly separated from the dominant negative manifold in the learned representation.

In [6]:

def create_local_error_maps(
    embedding,
    ground_truth,
    model_scores,
    pathology,
    label_indices_map,
    threshold=0.7,
    k=100,
    min_class_count=20
):

    # 1. Select pathology-specific labels and scores
    label_idx = label_indices_map[pathology]

    y_true = np.asarray(ground_truth[:, label_idx], dtype=float)
    y_scores = np.asarray(model_scores[:, label_idx], dtype=float)

    # 2. Convert scores into binary predictions
    y_pred = (y_scores >= threshold).astype(int)

    # 3. Exclude uncertain/missing labels and invalid coordinates
    valid = (
        np.isin(y_true, [0, 1])
        & np.isfinite(y_scores)
        & np.isfinite(embedding).all(axis=1)
    )

    # Keep all coordinates for neighborhood search.
    # Only valid binary labels contribute to the rates.
    coordinates = np.asarray(embedding, dtype=float)

    # 4. Calculate global error rates for comparison
    positives = valid & (y_true == 1)
    negatives = valid & (y_true == 0)

    global_fn = np.sum(positives & (y_pred == 0))
    global_tp = np.sum(positives & (y_pred == 1))

    global_fp = np.sum(negatives & (y_pred == 1))
    global_tn = np.sum(negatives & (y_pred == 0))

    global_fnr = global_fn / (global_fn + global_tp)
    global_fpr = global_fp / (global_fp + global_tn)

    # 5. Find k nearest neighbors for every point
    k = min(k, len(coordinates))

    nn = NearestNeighbors(n_neighbors=k)
    nn.fit(coordinates)

    neighbor_indices = nn.kneighbors(
        coordinates,
        return_distance=False
    )

    # 6. Calculate local FNR and FPR
    local_fnr = np.full(len(coordinates), np.nan)
    local_fpr = np.full(len(coordinates), np.nan)

    fn_counts = np.zeros(len(coordinates), dtype=int)
    positive_counts = np.zeros(len(coordinates), dtype=int)

    fp_counts = np.zeros(len(coordinates), dtype=int)
    negative_counts = np.zeros(len(coordinates), dtype=int)

    for i, neighbors in enumerate(neighbor_indices):

        # Only use neighbors with valid labels
        neighbor_valid = valid[neighbors]

        neighbor_true = y_true[neighbors]
        neighbor_pred = y_pred[neighbors]

        # Positive ground-truth cases
        pos = neighbor_valid & (neighbor_true == 1)

        n_pos = np.sum(pos)
        n_fn = np.sum(pos & (neighbor_pred == 0))

        positive_counts[i] = n_pos
        fn_counts[i] = n_fn

        if n_pos >= min_class_count:
            local_fnr[i] = n_fn / n_pos

        # Negative ground-truth cases
        neg = neighbor_valid & (neighbor_true == 0)

        n_neg = np.sum(neg)
        n_fp = np.sum(neg & (neighbor_pred == 1))

        negative_counts[i] = n_neg
        fp_counts[i] = n_fp

        if n_neg >= min_class_count:
            local_fpr[i] = n_fp / n_neg

    # 7. Helper function for plotting
    def plot_error_map(rate, error_counts, denominators,
                       global_rate, title):

        mask = valid & np.isfinite(rate)

        customdata = np.column_stack([
            error_counts[mask],
            denominators[mask]
        ])

        fig = go.Figure()

        fig.add_trace(go.Scatter(
            x=coordinates[mask, 0],
            y=coordinates[mask, 1],
            mode="markers",

            marker=dict(
                size=6,
                color=rate[mask],
                colorscale="RdYlBu_r",
                cmin=0,
                cmax=1,
                colorbar=dict(title="Local error rate"),
                opacity=0.85
            ),

            customdata=customdata,

            hovertemplate=(
                "Local error rate: %{marker.color:.1%}<br>"
                "Errors: %{customdata[0]}<br>"
                "Relevant cases: %{customdata[1]}<br>"
                "<extra></extra>"
            )
        ))

        fig.update_layout(
            title=(
                f"{pathology} – {title}<br>"
                f"<sup>Global rate: {global_rate:.1%} | "
                f"k={k} | Minimum denominator={min_class_count}</sup>"
            ),
            xaxis_title="S-UMAP 1",
            yaxis_title="S-UMAP 2",
            template="plotly_white",
            width=900,
            height=700
        )

        return fig

    # 8. Create the two maps
    fnr_fig = plot_error_map(
        local_fnr,
        fn_counts,
        positive_counts,
        global_fnr,
        "Local False Negative Rate"
    )

    fpr_fig = plot_error_map(
        local_fpr,
        fp_counts,
        negative_counts,
        global_fpr,
        "Local False Positive Rate"
    )

    print(f"Pathology: {pathology}")
    print(f"Threshold: {threshold}")
    print(f"Global FNR: {global_fnr:.2%}")
    print(f"Global FPR: {global_fpr:.2%}")

    fnr_fig.show()
    fpr_fig.show()

    return {
        "local_fnr": local_fnr,
        "local_fpr": local_fpr,
        "positive_counts": positive_counts,
        "negative_counts": negative_counts,
        "fn_counts": fn_counts,
        "fp_counts": fp_counts,
        "global_fnr": global_fnr,
        "global_fpr": global_fpr,
        "fnr_figure": fnr_fig,
        "fpr_figure": fpr_fig
    }


In [8]:

for pathology in XRAY_DATASET.pathologies:

    # Calculate the same S-UMAP embedding used in your
    # existing visualization.
    embedding = downprojector.calculate_projection(
        "feature",
        "S-UMAP",
        target_labels=XRAY_LABELS_LIST,
        label_name=pathology
    )

    results = create_local_error_maps(
        embedding=embedding,
        ground_truth=XRAY_LABELS_LIST,
        model_scores=MODEL_SCORE_PREDICTIONS_CONVERTED,
        pathology=pathology,
        label_indices_map=DATASET_LABEL_INDICES_MAP,
        threshold=0.7,
        k=100,
        min_class_count=20
    )


Pathology: Atelectasis
Threshold: 0.7
Global FNR: 37.00%
Global FPR: 37.37%


Pathology: Cardiomegaly
Threshold: 0.7
Global FNR: 28.64%
Global FPR: 22.62%


Pathology: Consolidation
Threshold: 0.7
Global FNR: 51.47%
Global FPR: 17.49%


Pathology: Edema
Threshold: 0.7
Global FNR: 37.14%
Global FPR: 21.37%


Pathology: Enlarged Cardiomediastinum
Threshold: 0.7
Global FNR: 69.33%
Global FPR: 14.83%


Pathology: Fracture
Threshold: 0.7
Global FNR: 87.11%
Global FPR: 9.43%


Pathology: Lung Lesion
Threshold: 0.7
Global FNR: 71.33%
Global FPR: 12.33%


Pathology: Lung Opacity
Threshold: 0.7
Global FNR: 30.99%
Global FPR: 29.87%


Pathology: Effusion
Threshold: 0.7
Global FNR: 28.43%
Global FPR: 21.10%


Pathology: Pneumonia
Threshold: 0.7
Global FNR: 76.68%
Global FPR: 12.38%


Pathology: Pneumothorax
Threshold: 0.7
Global FNR: 97.01%
Global FPR: 0.45%
